# 🌳 Cuaderno 02 — Newton y Lagrange

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

Tema 4.2. Dos formas del **mismo** polinomio, más la estimación del error.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
# ---------------------------------------------------------------
# En tu computadora usa los archivos locales, sin repositorio.
# En Google Colab pide UNA vez el enlace de tu fork, lo guarda en tu
# Drive y a partir de ahí lo reutiliza solo.
# ---------------------------------------------------------------
import json
import os
import subprocess
import sys

CUADERNO = 'u4_02_newton_lagrange'
TAREA_ID = 'U4_02_Newton_Lagrange'
REPO_NOMBRE = "U4_Repo"
GUARDADO = os.path.join("CursoMN", "u4_repo.json")   # relativo a MyDrive

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _raiz_local():
    """Sube desde la carpeta actual hasta encontrar lib/robomat.py."""
    candidatos = [os.getcwd()]
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        candidatos.append(os.path.dirname(os.path.abspath(nb)))
    for base in candidatos:
        b = base
        for _ in range(6):
            if os.path.exists(os.path.join(b, "lib", "robomat.py")):
                return b
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


def _normaliza(enlace):
    enlace = (enlace or "").strip().rstrip("/")
    if enlace.endswith(".git"):
        enlace = enlace[:-4]
    if not enlace.startswith("https://github.com/"):
        raise ValueError("Necesito un enlace https://github.com/USUARIO/"
                         + REPO_NOMBRE + " (tu fork).")
    return enlace


if not ES_COLAB:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError(
            "No encontré la raíz del repositorio (lib/robomat.py).\n"
            "Abre el cuaderno dentro de U4_Repo o ejecútalo desde esa carpeta.")
    print("Modo        : local (sin repositorio)")
    print("Repositorio :", REPO)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    ruta_json = os.path.join(DRIVE, GUARDADO)

    guardado = {}
    if os.path.exists(ruta_json):
        try:
            with open(ruta_json, encoding="utf-8") as f:
                guardado = json.load(f)
        except Exception:
            guardado = {}

    enlace = (guardado.get("repo") or "").strip()
    if enlace:
        print("✅ Repositorio guardado:", enlace)
        print("   (si cambió, edita la caja de abajo y pulsa el botón)")
    else:
        try:
            enlace = input("Pega el enlace de TU fork de U4_Repo: ")
        except Exception:
            enlace = ""
        enlace = (enlace or "").strip()
        if not enlace:
            raise RuntimeError(
                "Sin repositorio no puedo continuar en Colab.\n"
                "Haz fork del repositorio del curso y pega su enlace.")

    enlace = _normaliza(enlace)
    REPO = os.path.join(DRIVE, "CursoMN", REPO_NOMBRE)

    if not os.path.exists(REPO):
        print("Clonando tu repositorio…")
        subprocess.run(["git", "clone", enlace + ".git", REPO], check=True)
    else:
        print("Actualizando tu repositorio…")
        subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump({"repo": enlace}, f, indent=1)
    print("Repositorio :", REPO)

# --- Motor de calificación -------------------------------------------
# Se usa el código fuente si está (rama `test`, para depurar); si no, el blob
# empaquetado de grader/, que es lo que recibe el alumno.
_ruta_modulos = os.path.join(REPO, "lib", "mn_motor")
_ruta_blob = os.path.join(REPO, "grader", CUADERNO + ".txt")

_lib = os.path.join(REPO, "lib")
if _lib not in sys.path:
    sys.path.insert(0, _lib)          # ajuste_curvas y, si está, mn_motor

if os.path.isdir(_ruta_modulos):
    import mn_motor as _motor
    globals().update({n: getattr(_motor, n) for n in _motor.__all__})
    print("Motor       : módulos de lib/mn_motor (modo profesor)")
else:
    import base64
    import zlib

    if not os.path.exists(_ruta_blob):
        raise FileNotFoundError(
            "No encuentro el motor: falta grader/" + CUADERNO + ".txt\n"
            "Clona el repositorio completo (incluye la carpeta grader/).")
    with open(_ruta_blob, encoding="utf-8") as f:
        codigo = zlib.decompress(base64.b64decode(f.read().strip())).decode("utf-8")
    _MOTOR = {"__name__": "mn_motor_blob", "__file__": "<blob>"}
    exec(compile(codigo, "<blob>", "exec"), _MOTOR)
    globals().update({n: _MOTOR[n] for n in _MOTOR["__all__"]})
    print("Motor       : blob verificado de grader/")

usar_cuaderno(CUADERNO, TAREA_ID)

# --- Preludio estilo MATLAB y herramientas de gráficas ---------------
with open(os.path.join(REPO, "lib", "robomat.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

if REPO not in sys.path:
    sys.path.insert(0, REPO)          # para importar ejercicios/
sys.path.append(os.path.join(REPO, "tools"))
import importlib
import visualizar
importlib.reload(visualizar)

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
print("Preludio    : lib/robomat.py · tools/visualizar.py")
print("\nListo. Ejecuta la celda de identificación.")

In [ ]:
#@title 👤 Identificación (número de control)
# ---------------------------------------------------------------
# En Colab se toma tu correo institucional; en local se escribe uno
# de prueba. De ahí sale el número de control que siembra tus datos.
# ---------------------------------------------------------------
if ES_COLAB:
    import requests

    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    auth.authenticate_user()
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    info = requests.get(
        "https://www.googleapis.com/drive/v3/about?fields=user",
        headers={{"Authorization": "Bearer " + creds.token}}, timeout=20).json()
    cuenta = info.get("user", {{}})
    email = cuenta.get("emailAddress", "")
    nombre = cuenta.get("displayName", "")
else:
    email = "m16330887@hermosillo.tecnm.mx"   # <-- cambia por el tuyo
    nombre = ""

if not (email and "@" in email and email.lower().endswith(".tecnm.mx")):
    from IPython.display import HTML, display
    display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;'
        'border-radius:8px;font-weight:bold">⛔ Usa tu correo @….tecnm.mx</p>'))
    raise RuntimeError("Necesito tu correo institucional para sembrar tus datos.")

print("Alumno :", email)
print("NC     :", extraer_nc(email))
MI_TAREA = generar_tarea(email, nombre)
print("Cuaderno:", MI_TAREA["cuaderno"],
      "·", len(MI_TAREA["preguntas"]), "preguntas")

---

## Método 1 · Tabla de diferencias divididas

La primera diferencia dividida es una pendiente,

$$f[x_i,x_j]=\frac{f(x_i)-f(x_j)}{x_i-x_j} \tag{18.12}$$

y las siguientes se construyen con las anteriores:

$$f[x_i,x_j,x_k]=\frac{f[x_i,x_j]-f[x_j,x_k]}{x_i-x_k} \tag{18.13}$$

La tabla es triangular y su **diagonal** son los coeficientes del polinomio.

In [ ]:
#@title ✏️ Actividad a mano · Tabla de diferencias divididas
EJ_1 = mano_enunciado("NEWTON_TABLA")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Tabla de diferencias divididas
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("NEWTON_TABLA", [0.0, 0.0, 0.0, 0.0])

### 💻 Programa la función

Abre `ejercicios/newton_lagrange/tabla_diferencias.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Tabla de diferencias divididas
from ejercicios.newton_lagrange.tabla_diferencias import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_newton_tabla")

In [ ]:
#@title 📊 Laboratorio · Tabla de diferencias divididas
visualizar.laboratorio("NEWTON_TABLA", EJ_1)

In [ ]:
#@title 🧪 Pregunta 1 · Tabla de diferencias divididas
# Cuando tu función ya funcione, contéstala aquí.
pregunta(1)

---

## Método 2 · Polinomio de Newton en diferencias divididas

La forma general agrega un término por nodo:

$$f_n(x)=b_0+b_1(x-x_0)+\cdots+b_n(x-x_0)\cdots(x-x_{n-1}) \tag{18.7}$$

Se evalúa con Horner: $p=b_n$ y luego $p\leftarrow p(x-x_k)+b_k$. Esta función
**debe llamar** a `coeficientes_newton` y a `evaluar_newton`.

In [ ]:
#@title ✏️ Actividad a mano · Polinomio de Newton en diferencias divididas
EJ_2 = mano_enunciado("NEWTON_EVAL")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Polinomio de Newton en diferencias divididas
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("NEWTON_EVAL", [0.0])

### 💻 Programa la función

Abre `ejercicios/newton_lagrange/interpolacion_newton.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Polinomio de Newton en diferencias divididas
from ejercicios.newton_lagrange.interpolacion_newton import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_newton_eval")

In [ ]:
#@title 📊 Laboratorio · Polinomio de Newton en diferencias divididas
visualizar.laboratorio("NEWTON_EVAL", EJ_2)

In [ ]:
#@title 🧪 Pregunta 2 · Polinomio de Newton en diferencias divididas
# Cuando tu función ya funcione, contéstala aquí.
pregunta(2)

---

## Método 3 · Polinomio de Lagrange

Es una reformulación que evita las diferencias divididas:

$$f_n(x)=\sum_{i=0}^{n}L_i(x)f(x_i),\qquad
L_i(x)=\prod_{\substack{j=0\\ j\neq i}}^{n}\frac{x-x_j}{x_i-x_j} \tag{18.20–18.21}$$

Cada peso vale 1 en su nodo y 0 en los demás. Da **el mismo** polinomio que
Newton.

In [ ]:
#@title ✏️ Actividad a mano · Polinomio de Lagrange
EJ_3 = mano_enunciado("LAGRANGE")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Polinomio de Lagrange
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("LAGRANGE", [0.0])

### 💻 Programa la función

Abre `ejercicios/newton_lagrange/interpolacion_lagrange.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Polinomio de Lagrange
from ejercicios.newton_lagrange.interpolacion_lagrange import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_lagrange")

In [ ]:
#@title 📊 Laboratorio · Polinomio de Lagrange
visualizar.laboratorio("LAGRANGE", EJ_3)

In [ ]:
#@title 🧪 Pregunta 3 · Polinomio de Lagrange
# Cuando tu función ya funcione, contéstala aquí.
pregunta(3)

---

## Método 4 · Error del polinomio

El término que se agrega al subir de grado es una estimación del error
de grado $n$:

$$R_n=f_{n+1}(x)-f_n(x)=b_n\prod_{i<n}(x-x_i) \tag{18.19}$$

In [ ]:
#@title ✏️ Actividad a mano · Error del polinomio
EJ_4 = mano_enunciado("ERROR_NEWTON")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Error del polinomio
# Reporta los números que obtuviste, con 4 decimales.
mano_comprueba("ERROR_NEWTON", [0.0])

### 💻 Programa la función

Abre `ejercicios/newton_lagrange/error_newton.py` (en Codespaces) o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Error del polinomio
from ejercicios.newton_lagrange.error_newton import *

In [ ]:
#@title ▶️ Prueba tu función con el ejemplo genérico
practica("f_error_newton")

In [ ]:
#@title 📊 Laboratorio · Error del polinomio
visualizar.laboratorio("ERROR_NEWTON", EJ_4)

In [ ]:
#@title 🧪 Pregunta 4 · Error del polinomio
# Cuando tu función ya funcione, contéstala aquí.
pregunta(4)

In [ ]:
#@title ✅ 1 · Verificación de los ejercicios a mano
mano_comprueba_todos(["NEWTON_TABLA", "NEWTON_EVAL", "LAGRANGE", "ERROR_NEWTON"])

In [ ]:
#@title 📝 2 · Verificación de todas las preguntas (no envía nada)
puntos, maximo = calificar()

In [ ]:
#@title 📤 3 · Envío de las respuestas
enviar(email)

In [ ]:
#@title 🔎 4 · Verificación de lo que ya está enviado (no envía nada)
consultar_calificacion()